In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install faiss-cpu #install FAISS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.9 MB/s eta 0:00:00:00:0100:01


# Imports

In [3]:
import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [5]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Question 1: Predicted probability score assigned to the ground-truth correct option by the Zero Shot Classifier on row 150

In [6]:
zs_scores=zs(prompt_150,candidate_labels=labels_150)['scores']
ans_idx=zs(prompt_150,candidate_labels=labels_150)['labels'].index(ans_150)
print(f"Probability given to ground truth: {zs_scores[ans_idx]:.3f}")

Probability given to ground truth: 0.384


# Question 2

In [7]:
row_150_embeddings = np.expand_dims(model.encode(prompt_150, show_progress_bar=False),axis=0)
index.search(row_150_embeddings, k=10)

(array([[0.26394248, 0.26394248, 0.2664283 , 0.2664283 , 0.26862198,
         0.2699362 , 0.2699362 , 0.2699362 , 0.283161  , 0.28714603]],
       dtype=float32),
 array([[ 663, 1701, 1269, 1532,  576,  847, 1693, 1906,  168,  150]]))

In [8]:
print("Position of Correct Document:",index.search(row_150_embeddings, k=10)[1][0].tolist().index(150)+1)

Position of Correct Document: 10


# Question 3

In [9]:
retrieved_indices=index.search(row_150_embeddings, k=10)[1][0]
retrieved_indices

array([ 663, 1701, 1269, 1532,  576,  847, 1693, 1906,  168,  150])

In [10]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [11]:
print("Place of Correct Document:",np.argsort(-ce_scores).tolist().index(9)+1)

Place of Correct Document: 1


# Question 4

In [12]:
row_42=train.iloc[42]
prompt_42=row_42['prompt']

In [13]:
row_42_embeddings = np.expand_dims(model.encode(prompt_42, show_progress_bar=False),axis=0)
row_42_docs=index.search(row_42_embeddings, k=5)[1][0]
row_42_docs

array([ 42, 241, 439, 456, 506])

In [14]:
concatenated_docs=" ".join([kb[i] for i in row_42_docs])
concatenated_docs

'Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combi

In [15]:
string=f"Context: {concatenated_docs} Question: {prompt_42}"
string

'Context: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, o

In [16]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [17]:
print("Total Number of Tokens:",len(tokenizer(string,truncate=False)['input_ids']))

Total Number of Tokens: 216


# Question 5

In [18]:
true_document=kb[150]
string_150=f"Context: {true_document} Question: {prompt_150}"
string_150

'Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.'

In [19]:
ans_idx=zs(string_150,candidate_labels=labels_150)['labels'].index(ans_150)
zs_scores_q5=zs(string_150,candidate_labels=labels_150)['scores']
print(f"Probability given to ground truth: {zs_scores_q5[ans_idx]:.3f}")

Probability given to ground truth: 0.989


# Question 6

In [20]:
adv_string_150=f"Context: {kb[999]} Question: {prompt_150}"
adv_string_150

'Context: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics. Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.'

In [21]:
ans_idx=zs(adv_string_150,candidate_labels=labels_150)['labels'].index(ans_150)
zs_scores_q6=zs(adv_string_150,candidate_labels=labels_150)['scores']
print(f"Probability given to ground truth: {zs_scores_q6[ans_idx]:.3f}")

Probability given to ground truth: 0.529


# Question 7

In [22]:
hit_count=0
for _,row in train.iloc[:100].iterrows():
    prompt_row=row['prompt']
    ans_row=row[row['answer']]
    row_embeddings = np.expand_dims(model.encode(prompt_row, show_progress_bar=False),axis=0)
    row_docs=index.search(row_embeddings, k=5)[1][0]
    #print(ans_row)
    for i in row_docs:
        #print(kb[i])
        if kb[i]==ans_row:
            hit_count+=1
            break
        
print(f"Hit Rate: {hit_count:.1f}")

Hit Rate: 73.0


# Question 8

In [23]:
def map_at_3(true,preds):
    score=0.0
    for rank,pred in enumerate(preds,start=1):
        if pred==true:
            score=1.0/rank
            break
    return score

In [24]:
map3_scores=[]
for _,row in train[:20].iterrows():
    prompt_row=row['prompt']
    ans_row=row[row['answer']]
    row_embeddings = np.expand_dims(model.encode(prompt_row, show_progress_bar=False),axis=0)
    row_docs=index.search(row_embeddings, k=5)[1][0]

    docs_5 = [kb[i] for i in row_docs] 
    pairs = [[prompt_row, doc] for doc in docs_5] 
    ce_scores = cross_encoder.predict(pairs) 
    
    best_doc=kb[row_docs[np.argsort(-ce_scores)[0]]]
    aug_string=f"Context: {best_doc} Question: {prompt_row}"
    choices=["A","B","C","D","E"]
    row_labels=[row[i] for i in choices]
    zs_pred=zs(aug_string,candidate_labels=row_labels)
    zs_labels=zs_pred['labels'][:3]
    top3_preds=[]
    for i in zs_labels:
        top3_preds.append(choices[row_labels.index(i)])
    map3=map_at_3(row['answer'],top3_preds)
    map3_scores.append(map3)

map3_scores

[1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 0.5,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0]

In [25]:
print(f"Mean of all Map@3 Scores: {np.mean(map3_scores):.3f}")

Mean of all Map@3 Scores: 0.975
